In [ ]:
import mne
import numpy as np
import pandas as pd
import os
from scipy.signal import welch
from scipy.integrate import trapezoid
import warnings
warnings.filterwarnings('ignore')

data_path = r"C:\Users\hiro2\Documents\EEG_project\data\MUSIN-G"

behav_path = os.path.join(data_path, "stimuli", "Behavioural_data")
behav_df = pd.read_csv(behav_path, sep='\s+', engine='python')

# Muse S Athena対応チャンネル
# AF7=E51, AF8=E27, TP9=E89, TP10=E8
MUSE_CH = [50, 26, 88, 7]
MUSE_CH_NAMES = ['AF7(E51)', 'AF8(E27)', 'TP9(E89)', 'TP10(E8)']

def band_power(data, sf, band):
    freqs, psd = welch(data, fs=sf, nperseg=int(sf*2))
    idx = np.logical_and(freqs >= band[0], freqs <= band[1])
    return trapezoid(psd[:, idx], freqs[idx], axis=1)

records = []

for sub_id in range(1, 21):
    for ses_id in range(1, 13):
        set_path = os.path.join(
            data_path,
            f"sub-{sub_id:03d}",
            f"ses-{ses_id:02d}",
            "eeg",
            f"sub-{sub_id:03d}_ses-{ses_id:02d}_task-MusicListening_run-{ses_id}_eeg.set"
        )
        if not os.path.exists(set_path):
            print(f"✗ ファイルなし: sub-{sub_id:03d} ses-{ses_id:02d}")
            continue

        try:
            raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
            raw.filter(1., 40., fir_window='hamming', verbose=False)
            sf = raw.info['sfreq']
            data = raw.get_data()[MUSE_CH]  # 修正

            theta = band_power(data, sf, [4, 8])
            alpha = band_power(data, sf, [8, 13])
            beta  = band_power(data, sf, [13, 30])

            row = behav_df[(behav_df['Subject'] == sub_id) &
                           (behav_df['Song_ID'] == ses_id)]
            if row.empty:
                continue

            enjoyment   = row['Enjoyment'].values[0]
            familiarity = row['Familiarity'].values[0]

            rec = {
                'subject': sub_id,
                'song_id': ses_id,
                'enjoyment': enjoyment,
                'familiarity': familiarity,
                'theta_mean': theta.mean(),
                'alpha_mean': alpha.mean(),
                'beta_mean':  beta.mean(),
                'arousal':    beta.mean() / (alpha.mean() + 1e-20),
            }
            for i in range(len(MUSE_CH)):  # 修正
                rec[f'theta_{MUSE_CH_NAMES[i]}'] = theta[i]
                rec[f'alpha_{MUSE_CH_NAMES[i]}'] = alpha[i]
                rec[f'beta_{MUSE_CH_NAMES[i]}']  = beta[i]

            records.append(rec)
            print(f"✓ sub-{sub_id:03d} ses-{ses_id:02d} "
                  f"Enjoyment={enjoyment} Arousal={rec['arousal']:.3f}")

        except Exception as e:
            print(f"✗ エラー sub-{sub_id:03d} ses-{ses_id:02d}: {e}")

df = pd.DataFrame(records)
save_path = os.path.join(data_path, "..", "features.csv")
df.to_csv(save_path, index=False)
print(f"\n完了！{len(df)}件のレコードを保存")
print(df.describe())